In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
import os
import palettable
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import LearningRateMonitor, EarlyStopping, RichProgressBar, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger, CSVLogger
from loguru import logger
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from datetime import datetime

from machine_intelligence.models.GatedTransformer import GatedTransformer
from machine_intelligence.utilities.data_handling import load_parquet_table
from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule
from machine_intelligence.data.TransformsData import ZeroFill


In [ ]:
sample_index = 6

In [ ]:
cwd = Path.cwd()
cwd

In [ ]:
project_root = Path(cwd).parents[1]
project_root

In [ ]:
current_date = datetime.now().strftime('%Y-%m-%d')
plot_output_dir = project_root / 'plot_outputs' / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# model_path = project_root / Path("experiment_logs/gtn-debug/version_118/checkpoints/epoch=5-step=186.ckpt")

model_path = project_root / Path("machine_intelligence/models/trained_models/gtn_final-3-embedding/checkpoints/last.ckpt")
model = GatedTransformer.load_from_checkpoint(model_path, map_location="cpu")

In [ ]:
with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4_prev_0.3_imp_none"])

    print(cfg)

In [ ]:
wandb.init(mode="disabled")

In [ ]:

os.environ['PYTHONHASHSEED'] = str(42)
lit.seed_everything(42, workers=True)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)

In [ ]:
sepsis_pred_data.setup(stage="test")

inference_dl = sepsis_pred_data.test_dataloader()

In [ ]:
batch = next(iter(inference_dl))

In [ ]:
batch_features_ts, batch_features_stat, batch_labels = batch

In [ ]:
print(model)

In [ ]:
sample_ts = batch_features_ts[1].unsqueeze(0)
sample_stat = batch_features_stat[1].unsqueeze(0)
sample_label = batch_labels[1]


In [ ]:
sample_ts.shape

In [ ]:
model.to("cpu")
model.eval()

with torch.no_grad():
    pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        sample_ts, sample_stat, stage="test")


In [ ]:
score_input_sm.shape

Score channels has the shape num_heads x num_channels x num_channels

In [ ]:
score_channel.shape

In [ ]:
sns.heatmap(score_channel[0])

In [ ]:
sns.heatmap(score_channel_sm[3])

Average Normalized Attention across heads num_heads = 4

In [ ]:
sns.heatmap(torch.mean(score_channel_sm, dim=0))

Sum of the Attention Score Across Heads

In [ ]:
df_score_channel_sm_sum = pd.DataFrame(data=torch.sum(score_channel_sm, dim=0),
                                       columns=sepsis_pred_data.considered_features,
                                       index=sepsis_pred_data.considered_features)
df_score_channel_sm_sum.head()


In [ ]:
sns.heatmap(score_input_sm[3])

In [ ]:

df_score_sequence_sm_sum = pd.DataFrame(data=torch.sum(score_input_sm, dim=0))
# columns=sepsis_pred_data.considered_features,
# index=sepsis_pred_data.considered_features)
df_score_sequence_sm_sum.head()


In [ ]:
sns.heatmap(df_score_sequence_sm_sum)

## Time Series Attention of the 4 Attention Heads in the last encoder block

In [ ]:
from machine_intelligence.utilities.viz_setup import figure_setup, figure_setup_grid, five_min_to_hours
import matplotlib.ticker as ticker
import random


def plot_gtn_sequence_attention_heads_heatmap(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    fig, gs, axs, font_labels, font_annots = figure_setup_grid(1, 4, width_ratios=[0.9, 0.9, 0.9, 1.1],
                                                               height_ratios=[1])

    sns.heatmap(score_input_sm[0], ax=axs[0, 0], cbar=False, square=True)
    sns.heatmap(score_input_sm[1], ax=axs[0, 1], cbar=False, square=True)
    sns.heatmap(score_input_sm[2], ax=axs[0, 2], cbar=False, square=True)
    sns.heatmap(score_input_sm[3], ax=axs[0, 3], square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75})

    five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
    for ax in axs[0, 1:]:
        ax.set_yticks([])

    for ax in axs[0, :]:
        ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
        ax.xaxis.set_major_formatter(five_min_hour_formatter)
        ax.set_xlabel("Time [h]")

    for i, ax in enumerate(axs[0, :]):
        ax.set_title(f"Head {i + 1}")

    axs[0, 0].yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    axs[0, 0].yaxis.set_major_formatter(five_min_hour_formatter)
    axs[0, 0].set_xlabel("Time [h]")
    axs[0, 0].set_ylabel("Time [h]")
    
    axs[0,0].invert_yaxis()
    axs[0,1].invert_yaxis()
    axs[0,2].invert_yaxis()
    axs[0,3].invert_yaxis()
   

    return fig, axs



## Time Series Attention Averaged normalized Attention Weights of the last encoder block


In [ ]:
def plot_gtn_seq_attention_mean(model, batch_ts, batch_stat, sample_idx):
    """Sequence Level Attention averaged across attention heads"""
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    score_sequence_mean = torch.mean(score_input_sm, dim=0)

    fig_seq_attn_mean, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=8, height_cm=8, num_colors=11,
                                                                         height_ratios=[1])

    sns.heatmap(score_sequence_mean, ax=axs, square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75})

    five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
    # for ax in axs:
    #     ax.set_yticks([])

    axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    axs.xaxis.set_major_formatter(five_min_hour_formatter)
    axs.set_xlabel("Time [h]")

    # for i, ax in enumerate(axs[0,:]):
    #     ax.set_title(f"Head {i + 1}")

    axs.yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    axs.yaxis.set_major_formatter(five_min_hour_formatter)
    axs.set_xlabel("Time [h]")
    axs.set_ylabel("Time [h]")

    return fig_seq_attn_mean, axs

## Time Series Attention Averaged normalized Attention Score of the last encoder block

In [ ]:
def plot_gtn_seq_attention_unnorm_mean(model, batch_ts, batch_stat, sample_idx):
    """Sequence Level Attention averaged across attention heads"""
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    score_sequence_mean = torch.mean(score_input, dim=0)

    fig_seq_attn_mean, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=8, height_cm=8, num_colors=11,
                                                                         height_ratios=[1])

    sns.heatmap(score_sequence_mean, ax=axs, square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75})

    five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
    # for ax in axs:
    #     ax.set_yticks([])

    axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    axs.xaxis.set_major_formatter(five_min_hour_formatter)
    axs.set_xlabel("Time [h]")

    # for i, ax in enumerate(axs[0,:]):
    #     ax.set_title(f"Head {i + 1}")

    axs.yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    axs.yaxis.set_major_formatter(five_min_hour_formatter)
    axs.set_xlabel("Time [h]")
    axs.set_ylabel("Time [h]")

    return fig_seq_attn_mean, axs


In [ ]:
fig_seq_attn_mean, _ = plot_gtn_seq_attention_mean(model=model, batch_ts=batch_features_ts,
                                                   batch_stat=batch_features_stat, sample_idx=sample_index )

if save_as_svg := True:
    fig_seq_attn_mean.savefig(plot_output_dir / Path("gtn_attention_score_seq_mean.svg"), format="svg")


In [ ]:
fig_seq_attn_mean, _ = plot_gtn_seq_attention_unnorm_mean(model=model, batch_ts=batch_features_ts,
                                                          batch_stat=batch_features_stat, sample_idx=sample_index )


In [ ]:
def plot_gtn_sequence_attention_average_heatmap(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    fig, gs, axs, font_labels, font_annots = figure_setup(1, 4, width_ratios=[0.9, 0.9, 0.9, 1.1],
                                                          height_ratios=[1])

    sns.heatmap(score_input_sm[3], ax=axs[0, 3], square=True, cbar_kws={"label": "Attention Score"})

    five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
    for ax in axs[0, 1:]:
        ax.set_yticks([])

    for ax in axs[0, :]:
        ax.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
        ax.xaxis.set_major_formatter(five_min_hour_formatter)
        ax.set_xlabel("Time [h]")

    for i, ax in enumerate(axs[0, :]):
        ax.set_title(f"Head {i + 1}")

    axs[0, 0].yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    axs[0, 0].yaxis.set_major_formatter(five_min_hour_formatter)
    axs[0, 0].set_xlabel("Time [h]")
    axs[0, 0].set_ylabel("Time [h]")

    return fig, axs



In [ ]:
df_score_channel_sum_mean = df_score_channel_sm_sum.mean(axis=0)
df_score_channel_sum_mean_top20 = df_score_channel_sum_mean.sort_values(ascending=False).head(n=20)
df_score_ch_top20 = df_score_channel_sm_sum.loc[
    df_score_channel_sum_mean_top20.index, df_score_channel_sum_mean_top20.index]
df_score_ch_top20.head()

In [ ]:
from machine_intelligence.utilities.viz_setup import figure_setup_grid

from machine_intelligence.utilities.viz_setup import figure_setup, five_min_to_hours
import matplotlib.ticker as ticker
import random


def plot_gtn_channel_attention_ts_data(batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    df_sample_ts = pd.DataFrame(sample_ts.squeeze(0), columns=sepsis_pred_data.considered_features)
    zero_columns = df_sample_ts.loc[:, (df_sample_ts == 0).all()].columns.to_list()
    df_sample_ts = df_sample_ts.drop(columns=zero_columns)

    fig, axs, font_labels, font_annotations = figure_setup(2, 1, width_cm=14, height_cm=12, num_colors=11,
                                                           )

    print(df_score_ch_top20)
    ax0 = axs[0]
    ax1 = axs[1]
    markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', 'h', '*', '+', 'x', 'X', 'd']
    for col in df_sample_ts.columns:
        if "labname" in col:
            marker = random.choice(markers)
            sns.scatterplot(df_sample_ts[col], ax=ax1, label=col, marker=marker, s=50, zorder=100, alpha=0.95)
        else:
            sns.lineplot(df_sample_ts[col], ax=ax0, label=col)

    ax0.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.4),
        ncol=3,
        frameon=False,
        fancybox=True,
        shadow=True,
        prop=font_annotations,
    )
    ax1.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.4),
        ncol=3,
        frameon=False,
        fancybox=True,
        shadow=True,
        prop=font_annotations,
    )

    five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
    ax0.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax0.xaxis.set_major_formatter(five_min_hour_formatter)

    ax1.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax1.xaxis.set_major_formatter(five_min_hour_formatter)

    ax0.set_xlabel("Time [h]")
    ax1.set_xlabel("Time [h]")

    ax0.set_ylabel("Dense \n Time Series Variables",
                   )

    ax1.set_ylabel("Sparse \n Time Series Variables",
                   )
    
   
    ax0.margins(x=0)
    ax1.margins(x=0)

    # Adjust layout
    plt.tight_layout()
    plt.show()

    return fig, axs



In [ ]:
fig_attn_data, axs_attn_data = plot_gtn_channel_attention_ts_data(batch_ts=batch_features_ts,
                                                                  batch_stat=batch_features_stat,
                                                                  sample_idx=sample_index )

if save_as_svg := True:
    fig_attn_data.savefig(plot_output_dir / Path("gtn_attention_score_channel_data.svg"), format="svg")


In [ ]:
fig_attn_seq_heads_heat, axs_attn_seq_heads_heat = plot_gtn_sequence_attention_heads_heatmap(model=model,
                                                                                             batch_ts=batch_features_ts,
                                                                                             batch_stat=batch_features_stat,
                                                                                             sample_idx=sample_index )

if save_as_svg := True:
    fig_attn_seq_heads_heat.savefig(plot_output_dir / Path("gtn_attention_score_ts_heads.svg"), format="svg")


In [ ]:
def plot_gtn_channel_attention_heatmap(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    # df_score_channel_sm_sum = pd.DataFrame(data=torch.sum(score_channel_sm, dim=0),

    df_score_channel_sm_mean = pd.DataFrame(data=torch.mean(score_channel_sm, dim=0).numpy(), )
    # columns=sepsis_pred_data.considered_features,
    # index=sepsis_pred_data.considered_features)

    # df_score_channel_sum_mean = df_score_channel_sm_mean.mean(axis=0)
    df_score_channel_sum_mean_top20 = df_score_channel_sum_mean.sort_values(ascending=False).head(n=20)

    # df_score_ch_top20 = df_score_channel_sm_sum.loc[
    #     df_score_channel_sum_mean_top20.index, df_score_channel_sum_mean_top20.index]

    fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=14, height_cm=12, num_colors=11,
                                                           height_ratios=[1])

    # sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    # sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    sns.heatmap(df_score_channel_sm_mean, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
                xticklabels=1, yticklabels=1)

    axs.set_xlabel("Time Series Variables",
                   )

    axs.set_ylabel("Time Series Variables",
                   )

    axs.set_xticklabels(axs.get_xticklabels(), fontsize=5, rotation=90)
    axs.set_yticklabels(axs.get_yticklabels(), fontsize=5)

    # Adjust layout
    plt.tight_layout()
    plt.show()

    return fig, axs



In [ ]:
fig_attn_heat, axs_attn_heat = plot_gtn_channel_attention_heatmap(model=model, batch_ts=batch_features_ts,
                                                                  batch_stat=batch_features_stat,
                                                                  sample_idx=sample_index )

if save_as_svg := True:
    fig_attn_heat.savefig(plot_output_dir / Path("gtn_attention_score_channel.svg"), format="svg")


In [ ]:
xtick_lut = {}
for label, idx in enumerate(sepsis_pred_data.considered_features):
    xtick_lut[idx] = label

xtick_lut


## Attention Score unnormalized averaged over all heads

In [ ]:
def plot_gtn_channel_attention_unnorm_heatmap_all(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    df_score_channel_mean = pd.DataFrame(data=torch.mean(score_channel, dim=0).numpy(),
                                         columns=sepsis_pred_data.considered_features,
                                         index=sepsis_pred_data.considered_features)

    fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                           height_ratios=[1], share_y=False, share_x=False)

    # sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    # sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    sns.heatmap(df_score_channel_mean, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
                xticklabels=1, yticklabels=1)

    # axs.invert_xaxis()
    axs.invert_yaxis()

    axs.set_xlabel("Time Series Variables",
                   )

    axs.set_ylabel("Time Series Variables Bubats",
                   )

    axs.set_xticklabels(axs.get_xticklabels(), fontsize=4, rotation=90)
    axs.set_yticklabels(axs.get_yticklabels(), fontsize=4)

    # Adjust layout
    plt.tight_layout()
    plt.show()

    return fig, axs


plot_gtn_channel_attention_unnorm_heatmap_all(model=model, batch_ts=batch_features_ts, batch_stat=batch_features_stat,
                                              sample_idx=sample_index )

if save_as_svg := True:
    fig_attn_heat.savefig(plot_output_dir / Path("gtn_attention_score_channel_unnormalized_mean.svg"), format="svg")


## Attention Score normalized averaged over all heads

In [ ]:

def plot_gtn_channel_attention_norm_heatmap_all(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    df_score_channel_mean = pd.DataFrame(data=torch.mean(score_channel_sm, dim=0).numpy(),
                                         columns=sepsis_pred_data.considered_features,
                                         index=sepsis_pred_data.considered_features)

    fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                           height_ratios=[1], share_y=False, share_x=False)

    # sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    # sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    sns.heatmap(df_score_channel_mean, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
                xticklabels=1, yticklabels=1)

    # axs.invert_xaxis()
    axs.invert_yaxis()

    axs.set_xlabel("Time Series Variables",
                   )

    axs.set_ylabel("Time Series Variables Bubats",
                   )

    axs.set_xticklabels(axs.get_xticklabels(), fontsize=4, rotation=90)
    axs.set_yticklabels(axs.get_yticklabels(), fontsize=4)

    # Adjust layout
    plt.tight_layout()
    plt.show()

    return fig, axs


plot_gtn_channel_attention_norm_heatmap_all(model=model, batch_ts=batch_features_ts, batch_stat=batch_features_stat,
                                              sample_idx=sample_index)

if save_as_svg := True:
    fig_attn_heat.savefig(plot_output_dir / Path("gtn_attention_score_channel_normalized_mean.svg"), format="svg")


## Attention Score unnormalized averaged over all heads (threshold)

In [ ]:
def plot_gtn_channel_attention_heatmap_all_thresh(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    df_score_channel_mean = pd.DataFrame(data=torch.mean(score_channel, dim=0).numpy(),
                                         columns=sepsis_pred_data.considered_features,
                                         index=sepsis_pred_data.considered_features)

    threshold = 0.1
    df_thresholded = df_score_channel_mean.map(lambda x: x if x >= threshold else np.nan)

    df_score_channel_mean_thr = df_thresholded.dropna(axis=0, how='all').dropna(axis=1, how='all')
    fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                           height_ratios=[1])

    # sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    # sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    sns.heatmap(df_score_channel_mean_thr, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score",
                                                                                     "shrink": 0.25},
                xticklabels=1, yticklabels=1)

    axs.set_xlabel("Time Series Variables",
                   )

    axs.set_ylabel("Time Series Variables",
                   )

    axs.set_xticklabels(axs.get_xticklabels(),
                        # fontsize=4,
                        rotation=45,
                        ha="right",
                        )
    axs.set_yticklabels(axs.get_yticklabels(),
                        # fontsize=4
                        )

    plt.grid()
    plt.tight_layout()
    plt.show()

    return fig, axs


plot_gtn_channel_attention_heatmap_all_thresh(model=model, batch_ts=batch_features_ts, batch_stat=batch_features_stat,
                                              sample_idx=sample_index )

if save_as_svg := True:
    fig_attn_heat.savefig(plot_output_dir / Path("gtn_attention_score_channel_unnormalized_mean_thresholded.svg"),
                          format="svg")



## Same as above just normalized

Doesn't work as good for thresholding, since the softmax squashes the values even closer together and the differences get very small,
making it really hard to properly threshold the value

In [ ]:
def plot_gtn_channel_attention_heatmap_all_thresh(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    df_score_channel_mean = pd.DataFrame(data=torch.mean(score_channel_sm, dim=0).numpy(),
                                         columns=sepsis_pred_data.considered_features,
                                         index=sepsis_pred_data.considered_features)

    threshold = 0.015
    df_thresholded = df_score_channel_mean.map(lambda x: x if x >= threshold else np.nan)

    df_score_channel_mean_thr = df_thresholded.dropna(axis=0, how='all').dropna(axis=1, how='all')
    fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                           height_ratios=[1])

    # sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    # sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    sns.heatmap(df_score_channel_mean_thr, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score",
                                                                                     "shrink": 0.25},
                xticklabels=1, yticklabels=1)

    axs.set_xlabel("Time Series Variables",
                   )

    axs.set_ylabel("Time Series Variables",
                   )

    axs.set_xticklabels(axs.get_xticklabels(),
                        # fontsize=4,
                        rotation=45,
                        ha="right",
                        )
    axs.set_yticklabels(axs.get_yticklabels(),
                        # fontsize=4
                        )

    plt.grid()
    plt.tight_layout()
    plt.show()

    return fig, axs

plot_gtn_channel_attention_heatmap_all_thresh(model=model, batch_ts=batch_features_ts, batch_stat=batch_features_stat,
                                               sample_idx=6)



Elementwise Multiplication Time Series & Channel Pattern



In [ ]:

sample_idx = 0
sample_ts = batch_features_ts[sample_idx].unsqueeze(0)
sample_stat = batch_features_stat[sample_idx].unsqueeze(0)

model.to("cpu")
model.eval()
with torch.no_grad():
    pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        sample_ts, sample_stat, stage="test")



In [ ]:
attention_score_channel = torch.mean(score_channel_sm, dim=0).numpy()
attention_score_ts = torch.mean(score_input_sm, dim=0).numpy()

In [ ]:
attention_score_ts_mean_query = np.mean(attention_score_ts, axis=0)
attention_score_ts_mean_key = np.mean(attention_score_ts, axis=1)

In [ ]:
fig, axs = plt.subplots(1,1)
sns.heatmap(attention_score_ts)

# axs.invert_xaxis()



In [ ]:

sns.heatmap(np.expand_dims(attention_score_ts_mean_query, 0), square=False)


In [ ]:

fig_seq_attn_mean, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=8, height_cm=8, num_colors=11,
                                                                     height_ratios=[1])

sns.heatmap(np.expand_dims(attention_score_ts_mean_query, 0), ax=axs, square=True, cbar_kws={"label": "Attention Score", "shrink": 0.75})

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
# for ax in axs:
#     ax.set_yticks([])

axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)
axs.set_xlabel("Time [h]")

# for i, ax in enumerate(axs[0,:]):
#     ax.set_title(f"Head {i + 1}")

axs.yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.yaxis.set_major_formatter(five_min_hour_formatter)

In [ ]:
fig_seq_attn_mean_line, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=8, height_cm=8, num_colors=11,
                                                                     height_ratios=[1])

sns.lineplot(np.expand_dims(attention_score_ts_mean_query, 0).T, ax=axs)

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
# for ax in axs:
#     ax.set_yticks([])

axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)
axs.set_xlabel("Time [h]")

# for i, ax in enumerate(axs[0,:]):
#     ax.set_title(f"Head {i + 1}")

axs.yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.yaxis.set_major_formatter(five_min_hour_formatter)


key-wise mean makes no sense, see below

In [ ]:

sns.heatmap(np.expand_dims(attention_score_ts_mean_key, 0))


In [ ]:
sns.heatmap(attention_score_channel)

In [ ]:

attention_score_channel_mean_query = np.mean(attention_score_channel, axis=0)
attention_score_channel_mean_key = np.mean(attention_score_channel, axis=1)


In [ ]:
sns.heatmap(np.expand_dims(attention_score_channel_mean_query, 0))


In [ ]:

sns.heatmap(np.expand_dims(attention_score_channel_mean_key, 0))


In [ ]:

# attention_fused = np.multiply(attention_score_ts, attention_score_channel) 
np.multiply(sample_ts, attention_score_channel[0])

In [ ]:
sns.heatmap(np.multiply(sample_ts, attention_score_channel[0]).squeeze(0).T)

In [ ]:
df_sample_ts = pd.DataFrame(data=sample_ts.squeeze(0), columns=sepsis_pred_data.considered_features)

sns.heatmap(df_sample_ts.T)

In [ ]:
ones_ts = np.ones(shape=(72, 69)) * df_sample_ts
sns.heatmap(ones_ts.T)

In [ ]:
# attention_ones = np.multiply(attention_score_ts, np.ones(shape=(72, 69)))
# attention_ones = attention_score_ts * np.ones(shape=(72, 69)) * attention_score_channel
attention_ts_ones =  np.dot(attention_score_ts, np.ones(shape=(72, 69)) )

sns.heatmap(attention_ts_ones)

In [ ]:
attention_ts_ones.shape

In [ ]:
attention_ones_both = np.dot(attention_ts_ones, attention_score_channel)
sns.heatmap(attention_ones_both)

In [ ]:
attention_ts_feature_weighted = np.outer(attention_score_ts_mean_query, attention_score_channel_mean_query)
attention_ts_feature_weighted.shape

In [ ]:
df_attention_ts_feature_weighted = pd.DataFrame(data=attention_ts_feature_weighted, columns=sepsis_pred_data.considered_features)

In [ ]:
df_sample_ts

In [ ]:
# df_sample_ts = pd.DataFrame(sample_ts.squeeze(0), columns=sepsis_pred_data.considered_features)
zero_columns = df_sample_ts.loc[:, (df_sample_ts == 0).all()].columns.to_list()
df_sample_ts_present = df_sample_ts.drop(columns=zero_columns)
df_sample_ts_present

In [ ]:
# sns.heatmap(attention_ts_feature_weighted.T, cmap=sns.color_palette("magma", as_cmap=True))
fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                       height_ratios=[1], share_y=False, share_x=False)

df_attention_ts_feature_weighted_present = df_attention_ts_feature_weighted.loc[:,df_sample_ts_present.columns]
# sns.heatmap(df_attention_ts_feature_weighted.T, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
sns.heatmap(df_attention_ts_feature_weighted_present.T, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
            xticklabels=1, yticklabels=1)

# axs.invert_xaxis()
# axs.invert_yaxis()


axs.set_ylabel("Time Series Variables",
               )

five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
axs.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
axs.xaxis.set_major_formatter(five_min_hour_formatter)


axs.set_xlabel("Time [h]")
axs.margins(x=0)

# axs.set_xticklabels(axs.get_xticklabels(), fontsize=4, rotation=90)
axs.set_xticklabels(axs.get_xticklabels(), rotation=0)
axs.set_yticklabels(axs.get_yticklabels(), fontsize=4)

# Adjust layout
plt.tight_layout()
plt.show()


if save_as_svg := True:
    fig.savefig(plot_output_dir / Path("gtn_attention_mixed.svg"), format="svg")
    



In [ ]:

def plot_gtn_fused_attention_heatmap(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")
    
    
    attention_score_channel = torch.mean(score_channel_sm, dim=0).numpy()
    attention_score_ts = torch.mean(score_input_sm, dim=0).numpy()
    
    attention_fused = np.multiply(attention_score_ts, attention_score_channel) 
    
    
    # df_score_channel_mean = pd.DataFrame(data=torch.mean(score_channel_sm, dim=0).numpy(),
    #                                      columns=sepsis_pred_data.considered_features,
    #                                      index=sepsis_pred_data.considered_features)


    fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=15, height_cm=12, num_colors=11,
                                                           height_ratios=[1], share_y=False, share_x=False)
 # sns.heatmap(df_score_ch_top20, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    # sns.heatmap(df_score_channel_sm_mean, ax=axs, cbar=True, cbar_kws={"label": "Attention Score"})
    sns.heatmap(df_score_channel_mean, ax=axs, square=True, cbar=True, cbar_kws={"label": "Attention Score"},
                xticklabels=1, yticklabels=1)

    # axs.invert_xaxis()
    axs.invert_yaxis()

    axs.set_xlabel("Time Series Variables",
                   )

    axs.set_ylabel("Time Series Variables Bubats",
                   )

    axs.set_xticklabels(axs.get_xticklabels(), fontsize=4, rotation=90)
    axs.set_yticklabels(axs.get_yticklabels(), fontsize=4)

    # Adjust layout
    plt.tight_layout()
    plt.show()

    return fig, axs

plot_gtn_channel_attention_heatmap_all_thresh(model=model, batch_ts=batch_features_ts, batch_stat=batch_features_stat,
                                              sample_idx=6)



Channel Level Attention Score Bars, don't make as much sense, since the attention score is determined pairwise, needs to be averaged in both dims

def plot_gtn_channel_attention_bars(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()
    with torch.no_grad():
        pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
            sample_ts, sample_stat, stage="test")

    df_score_channel_mean = pd.DataFrame(data=[torch.mean(score_channel_sm, dim=(0, 1)).numpy()],
                                         columns=sepsis_pred_data.considered_features, index=["score"])

    df_score_channel_mean_top20 = df_score_channel_mean.sort_values(ascending=False, by="score", axis=1).T.head(n=20)

    fig, axs, font_labels, font_annotations = figure_setup(1, 1, width_cm=12, height_cm=12, num_colors=11,
                                                           )

    sns.barplot(df_score_channel_mean_top20.T, ax=axs, palette=palettable.matplotlib.Plasma_20.mpl_colors)

    axs.set_xlabel("Time Series Variables",
                   )

    axs.set_ylabel("Attention Score",
                   )

    axs.set_xticklabels(axs.get_xticklabels(), rotation=45, ha="right")

    # # Adjust layout
    plt.tight_layout()
    plt.show()

    return fig, axs


fig_channel_attn_bars_mean_heads, _  = plot_gtn_channel_attention_bars(model=model, batch_ts=batch_features_ts, batch_stat=batch_features_stat, sample_idx=6)

if save_as_svg := True:
    fig_channel_attn_bars_mean_heads.savefig(plot_output_dir / Path("gtn_attention_channel_mean_heads_bars.svg"), format="svg")


Contribution Time Series Tower vs. Channel Tower 

In [ ]:
model.to("cpu")
model.eval()
with torch.no_grad():
    pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
        sample_ts, sample_stat, stage="test")

In [ ]:
channel_head_mean = torch.mean(score_channel_sm, dim=0).squeeze(0)
channel_mean_sum = channel_head_mean.sum(0)
channel_mean_sum.shape

In [ ]:
gather_input.shape

In [ ]:
gather_channel.shape

## Averaging the the channel level attention pattern over multiple patients

In [ ]:
from tqdm.auto import tqdm
def aggregate_attention_weights(data_module: lit.LightningDataModule, model, device="cpu"):
    attention_weights = []
    attention_data = []

    model.eval()
    model.to(device)

    test_dataloader = data_module.test_dataloader()

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Aggregating Attention Weights"):
            sample_ts, sample_stat, sample_label = batch

            sample_ts = sample_ts.to(device)
            sample_stat = sample_stat.to(device)

            pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
                sample_ts, sample_stat, stage="test")


            # attention_weights.append(score_channel_sm.cpu())
            attention_weights.append(score_channel_sm)
            
            threshold = 0.5
            preds_probabilities = torch.sigmoid(pred).squeeze(1).to("cpu")
            preds = (preds_probabilities >= threshold).int()
            # pred_label = torch.argmax(pred, dim=1).to("cpu")
            correct = (preds == sample_label).to("cpu").numpy()
           
            average_attention_weights = torch.mean(score_channel_sm, dim=0)
            
            for i in range( int(score_channel_sm.size(0) / 4)): # n_heads = 4 (we get a score for each head)
                # attention_data.append((score_channel_sm[i], preds[i].item(), sample_label[i].item(), correct[i]))
                attention_data.append((average_attention_weights, preds[i].item(), sample_label[i].item(), correct[i]))
    
    
    
    
    # test_dataloader.teardown(stage="test")
    # attention_weights = torch.cat(attention_weights, dim=0)

    # average_attention_weights = torch.mean(attention_weights, dim=0)

    return attention_weights, attention_data 


sample_ts, sample_stat, sample_label = batch

# sample_ts = sample_ts.to(device)
# sample_stat = sample_stat.to(device)
model.eval()
model.to("cpu")

pred, enc, score_input, score_input_sm, score_channel, score_channel_sm, gather_input, gather_channel, gate = model(
    sample_ts, sample_stat, stage="test")


# attention_weights.append(score_channel_sm.cpu())
# attention_weights.append(score_channel_sm)

threshold = 0.5
preds_probabilities = torch.sigmoid(pred).squeeze(1).to("cpu")
preds = (preds_probabilities >= threshold).int()
# pred_label = torch.argmax(pred, dim=1).to("cpu")
correct = (preds == sample_label).to("cpu").numpy()

In [ ]:
attention_score_all_dl, attention_data_dl  = aggregate_attention_weights(data_module=sepsis_pred_data, model=model, device="cuda")

In [ ]:
attention_data_dl[0] # is the wrong length, debug later

In [ ]:
len(attention_data_dl)

In [ ]:
print(len(next(iter(sepsis_pred_data.test_dataloader()))))

In [ ]:
print("hall")

6116